# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-9862873/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This notebook trains a model to predict content decline, compares it against the Week-4 baseline on the same split and metric, and reads the errors.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `training-honest-models` + `flyrank/flyrank-data` for this task.

## 1. Method choice and why

I use Logistic Regression first because:
- The question is yes/no (declining or not) with an observed label
- Logistic Regression is readable — coefficients tell you direction and magnitude
- It handles linear boundaries well, and the signals (staleness, volume) may combine linearly
- I start here before trying anything more complex

If Logistic Regression does not beat the baseline, I try Random Forest.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f'Rows: {len(df):,}, Base rate (declining): {df["is_declining_label"].mean():.1%}')

Rows: 30,000, Base rate (declining): 54.2%


## 2. Split design

Grouped train/test split by client_id. 80/20, random_state=42. This ensures the model generalizes to unseen clients, not just unseen pages from known clients.

In [2]:
# Features: only observed signals, no label-derived columns
feature_cols = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'content_age_days',
    'days_since_last_update', 'word_count'
]

df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_position'] = (df['avg_position'] > 0).astype(int)
feature_cols += ['has_word_count', 'has_position']

X = df[feature_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

print(f'Train: {len(X_train):,}, Test: {len(X_test):,}')
print(f'Train clients: {groups.iloc[tr_idx].nunique()}, Test clients: {groups.iloc[te_idx].nunique()}')
print(f'Train base rate: {y_train.mean():.3f}, Test base rate: {y_test.mean():.3f}')

Train: 23,837, Test: 6,163
Train clients: 25, Test clients: 7
Train base rate: 0.550, Test base rate: 0.511


## 3. Train + compare vs my baseline

Same data, same split, same metric.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Recompute baseline on the same test set
test_df = df.iloc[te_idx].copy()
stale = (test_df['days_since_last_update'] >= 180).astype(int)
visible = (test_df['impressions_90d'] >= 500).astype(int)
baseline_score = stale * visible * test_df['impressions_90d']

baseline_p10 = precision_at_k(baseline_score.values, y_test.values, 10)
baseline_p20 = precision_at_k(baseline_score.values, y_test.values, 20)
baseline_p50 = precision_at_k(baseline_score.values, y_test.values, 50)

# Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]

lr_p10 = precision_at_k(lr_prob, y_test.values, 10)
lr_p20 = precision_at_k(lr_prob, y_test.values, 20)
lr_p50 = precision_at_k(lr_prob, y_test.values, 50)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]

rf_p10 = precision_at_k(rf_prob, y_test.values, 10)
rf_p20 = precision_at_k(rf_prob, y_test.values, 20)
rf_p50 = precision_at_k(rf_prob, y_test.values, 50)

results = pd.DataFrame({
    'method': ['Baseline (rule)', 'Logistic Regression', 'Random Forest'],
    'precision@10': [baseline_p10, lr_p10, rf_p10],
    'precision@20': [baseline_p20, lr_p20, rf_p20],
    'precision@50': [baseline_p50, lr_p50, rf_p50],
    'f1': ['-', f'{f1_score(y_test, lr.predict(X_test)):.3f}', f'{f1_score(y_test, rf.predict(X_test)):.3f}'],
    'auc': ['-', f'{roc_auc_score(y_test, lr_prob):.3f}', f'{roc_auc_score(y_test, rf_prob):.3f}']
})
print(f'Base rate (test): {y_test.mean():.3f}')
results

Base rate (test): 0.511


,method,precision@10,precision@20,precision@50,f1,auc
0,Baseline (rule),0.5,0.45,0.62,-,-
1,Logistic Regression,0.6,0.70,0.68,0.644,0.573
2,Random Forest,0.7,0.70,0.60,0.621,0.603


## 4. Errors and interpretation

Where is the model wrong? What does it lean on?

In [4]:
# Feature importance (Random Forest)
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print('Top 5 features (Random Forest):')
importances.head(5)

Top 5 features (Random Forest):


,feature,importance
0,impressions_90d,0.241024
4,avg_position,0.189138
7,content_age_days,0.153102
9,word_count,0.098580
11,has_position,0.057850


In [5]:
# Logistic Regression coefficients
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print('Top coefficients (Logistic Regression):')
coef_df.head(5)

Top coefficients (Logistic Regression):


,feature,coefficient
11,has_position,1.204336
7,content_age_days,-0.305949
5,ctr,-0.260004
8,days_since_last_update,0.172762
4,avg_position,-0.160212


In [6]:
# Error analysis
test_df = df.iloc[te_idx].copy()
test_df['pred'] = rf.predict(X_test)
test_df['prob'] = rf_prob
test_df['correct'] = (test_df['pred'] == test_df['is_declining_label']).astype(int)

fp = test_df[(test_df['pred'] == 1) & (test_df['is_declining_label'] == 0)]
fn = test_df[(test_df['pred'] == 0) & (test_df['is_declining_label'] == 1)]

print(f'False positives: {len(fp):,} ({len(fp)/len(test_df)*100:.1f}%)')
print(f'False negatives: {len(fn):,} ({len(fn)/len(test_df)*100:.1f}%)')

if len(fp) > 0:
    print(f'FP: median impressions={fp["impressions_90d"].median():.0f}, '
          f'median position={fp["avg_position"].median():.1f}')
if len(fn) > 0:
    print(f'FN: median impressions={fn["impressions_90d"].median():.0f}, '
          f'median position={fn["avg_position"].median():.1f}')

False positives: 1,609 (26.1%)
False negatives: 1,004 (16.3%)
FP: median impressions=549, median position=10.7
FN: median impressions=608, median position=12.1


In [7]:
# 3 concrete wrong cases
wrong = test_df[test_df['correct'] == 0].head(3)
for _, row in wrong.iterrows():
    label = 'declining' if row['is_declining_label'] == 1 else 'stable'
    pred = 'declining' if row['pred'] == 1 else 'stable'
    print(f"{row['content_id']}: actual={label}, predicted={pred}, "
          f"prob={row['prob']:.3f}, impressions={int(row['impressions_90d'])}, "
          f"position={row['avg_position']:.1f}, stale={int(row['days_since_last_update'])}d")

content_a5a2fbc76336: actual=stable, predicted=declining, prob=0.819, impressions=307, position=39.8, stale=103d
content_2da6ae9d0882: actual=declining, predicted=stable, prob=0.482, impressions=297, position=13.9, stale=20d
content_72c5c2d73e5a: actual=stable, predicted=declining, prob=0.734, impressions=2426, position=30.0, stale=13d


### What the errors tell us

The model struggles with:
1. Pages with moderate impressions where decline and stability overlap
2. The stale flag drives both the baseline and the model — when it is wrong, both are wrong
3. Low-impression pages where the signal is too weak to distinguish decline from noise

## Self-check

- [x] Every section above is filled
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words
- [x] Committed to my repo under `work/notebooks/`